## Imports

In [ ]:
# Copyright 2026 András Biricz. Licensed under the Apache License, Version 2.0.
# Restrict CPU usage to specific threads
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import glob
import re

# System and I/O
import sys
import io
import json
import argparse  # Argument parsing
import warnings
from collections import defaultdict, Counter
from contextlib import redirect_stdout

# Data handling and processing
import random
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelBinarizer, LabelEncoder
from sklearn.metrics import f1_score, precision_recall_curve, classification_report
import math

# PyTorch and deep learning
import torch
torch.backends.cudnn.benchmark = True
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR, SequentialLR
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import transforms
#from torchvision.io import read_image
#from torchvision.models import resnet50, ResNet50_Weights
#from torchvision import models as torchvision_models
#from torch.optim import lr_scheduler
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FasterRCNN_ResNet50_FPN_V2_Weights

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageFilter
from tqdm import tqdm
import cv2
import skimage.io as skio
import albumentations as A
from skimage.measure import shannon_entropy
from skimage.metrics import structural_similarity as ssim
#from albumentations.pytorch import ToTensorV2

# Stain normalization
from tiatoolbox.tools.stainnorm import MacenkoNormalizer

# Transformers and pretrained models
import timm
#from transformers import AutoModel, AutoImageProcessor
import umap
import seaborn as sns
from sklearn.cluster import DBSCAN

# Configuration
Image.MAX_IMAGE_PIXELS = None  # Disable PIL image size restrictions
warnings.filterwarnings("ignore")  # Ignore warnings

from datetime import datetime

In [ ]:
CUDANUM = 0
device = torch.device(f'cuda:{CUDANUM}') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
!ls <DATA_ROOT>/ | grep images

## General assembly

In [ ]:
slidenames = ['Alnus_cf_incana_40x_11_steps_merged_reference',
        'Ambrosia-Iva_reference_10l_1m_Ambrosia_edf',
        'Ambrosia-Iva_reference_10l_1m_Iva_edf',
        'Betula_cf_pendula_40x_11_steps_merged_reference',
        'Corylus_avellana_40x_11_steps_merged_reference',
        'Pinus_sp_10_Pinaceae_14_layers_40x_Blue_colou_ZS017_5_mm_circle',
        'Quercus_robur_Fagaceae_14_layers_40x_check_quality_ZS017_5_mm_circle',
        'Salix_sp_10_Salicaceae_14_layers_40x_ZS017_5_mm_circle',
        'Urtica_dioica_40x_12_layers_03_crowded_ZS015_5_mm_circle',
        'acer_edf', 'ambrosia_edf', 'betula_2_edf', 'betula_edf',
        'brassica_napus_edf', 'brassicaceae_2_edf', 'brassicaceae_edf',
        'cannabis_edf', 'carpinus_edf', 'cedrus_2_edf', 'cedrus_edf',
        'cheno_edf', 'corylus_2_edf', 'corylus_edf', 'cupressus_edf',
        'ericaceae_edf', 'fabaceae_edf', 'festuca_edf', 'forsythia_edf',
        'fraxinus_edf', 'ginkgo_edf', 'hedera_edf',
        'humulus_japonicus_2_edf', 'humulus_japonicus_edf',
        'hun_betula_edf', 'hun_corylus_edf', 'juglans_edf',
        'juncaceae_edf', 'ligustrum_2_edf', 'ligustrum_edf',
        'mediterranean_pollen_causarina_reference',
        'mediterranean_pollen_cheno_reference',
        'mediterranean_pollen_cupr_reference',
        'mediterranean_pollen_olea_reference',
        'mediterranean_pollen_palmaceae_reference',
        'mediterranean_pollen_pinus_reference',
        'mediterranean_pollen_plantago_reference',
        'mediterranean_pollen_platanus_reference',
        'mediterranean_pollen_poaceae_reference',
        'mediterranean_pollen_rumex_reference',
        'mediterranean_pollen_urti_reference', 'mimosa_edf',
        'morus_alba_edf', 'parietaria_edf', 'phacelia_edf', 'picea_edf',
        'pinus_2_edf', 'plantago_edf', 'platanus_edf', 'quercus_edf',
        'ranunculus_edf', 'rumex_edf', 'salix_edf', 'tilia_edf',
        'triticum_aestivum_edf', 'typha_edf', 'ulmus_edf', 'urtica_edf',
        'vitis_edf']

classes = ['alnus',
        'ambrosia',
        'iva',
        'betula',
        'corylus',
        'pinus',
        'quercus',
        'salix',
        'urtica',
        'acer', 'ambrosia', 'betula', 'betula',
        'brassica_napus', 'brassicaceae', 'brassicaceae',
        'cannabis', 'carpinus', 'cedrus', 'cedrus',
        'cheno', 'corylus', 'corylus', 'cupressus',
        'ericaceae', 'fabaceae', 'festuca', 'forsythia',
        'fraxinus', 'ginkgo', 'hedera',
        'humulus_japonicus', 'humulus_japonicus',
        'hun_betula', 'hun_corylus', 'juglans',
        'juncaceae', 'ligustrum_2', 'ligustrum',
        'causarina',
        'cheno',
        'cupr',
        'olea',
        'palmaceae',
        'pinus',
        'plantago',
        'platanus',
        'poaceae',
        'rumex',
        'urti', 'mimosa',
        'morus_alba', 'parietaria', 'phacelia', 'picea',
        'pinus', 'plantago', 'platanus', 'quercus',
        'ranunculus', 'rumex', 'salix', 'tilia',
        'triticum_aestivum', 'typha', 'ulmus', 'urtica',
        'vitis']

slidenames_to_classes = dict(zip(slidenames, classes))

In [ ]:
def collect_images_and_labels_with_species(base_folders):
    images = []
    binary_labels = []
    species_list = []

    for base_folder in base_folders:
        pattern = os.path.join(base_folder, "owlvit-base-patch32", "*", "*", "*.png")
        all_files = glob.glob(pattern, recursive=True)

        for file_path in all_files:
            parts = file_path.split(os.sep)

            # Extract species and class (pollen_grains/negative_samples)
            species = parts[-3]  # folder name, e.g., "quercus_edf"
            class_type = parts[-2]  # either "pollen_grains" or "negative_samples"

            images.append(file_path)
            species_list.append(species)

            if class_type == "pollen_grains":
                binary_labels.append("pollen_grains")  # Multi-class: species name as label
            else:
                binary_labels.append("background")  # All negatives into 1 class

    return np.array(images), np.array(binary_labels), np.array(species_list)

In [ ]:
def unify_species_name(raw_species, slidenames_to_classes):
    """
    Maps raw species names to their unified class names using a predefined dictionary.
    
    Args:
        raw_species (str): Raw species name (slide name, folder name).
        slidenames_to_classes (dict): Dictionary mapping slide/folder names to target class names.
        
    Returns:
        str: Unified species class name, or "unknown" if no match is found.
    """
    raw_species = raw_species
    
    # Direct lookup
    return slidenames_to_classes.get(raw_species, "unknown")

### Locate all datasets

In [ ]:
parent_folder = '<DATA_ROOT>/'

base_folders = [
    "images_french/",
    "images_hungarian/",
    "images_mediterranean/",
    "images_swedish/"
]
base_folders = [ parent_folder+k for k in base_folders ]
base_folders

In [ ]:
!ls <DATA_ROOT>/images_french/owlvit-base-patch32/acer_edf/pollen_grains | grep .png | tail

In [ ]:
# Load all
images_all, labels_all, species_all = collect_images_and_labels_with_species(base_folders)

In [ ]:
print(f"Total images: {len(images_all)}")
print(f"Unique species: {set(species_all)}")
print(f"Label counts: {dict(zip(*np.unique(labels_all, return_counts=True)))}")

In [ ]:
unified_species = np.array( [unify_species_name(s, slidenames_to_classes) for s in species_all] )
uqs_species, cnts = np.unique( unified_species, return_counts=True )
uqs_species, cnts

In [ ]:
classes_to_int = dict(zip(uqs_species, np.arange(uqs_species.shape[0]) ))

#### check

In [ ]:
np.unique( np.array(species_all)[ (np.array(unified_species) == 'unknown')] )

In [ ]:
filt = labels_all == 'pollen_grains'
filt.sum()

### Create datasets and loaders

#### Dataset 

In [ ]:
transform = transforms.Compose([
    transforms.Resize((518, 518)),   # Adapt to ViT expected input size
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
class PollenValidationDataset(Dataset):
    def __init__(self, image_paths, labels, classes_to_int, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.classes_to_int = classes_to_int
        self.transform = transform

        print(f"Validation dataset with {len(self.image_paths)} images across {len(set(self.labels))} species.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        # Convert label to integer
        label = self.classes_to_int[label]

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        return {"image": image, "label": torch.tensor(label, dtype=torch.long)}

class BalancedPollenDataset(Dataset):
    def __init__(self, image_paths, labels, classes_to_int, transform=None, seed=None):
        self.transform = transform

        if seed is not None:
            np.random.seed(seed)
            random.seed(seed)
            print(f"BalancedPollenDataset initialized with seed: {seed}")

        self.classes_to_int = classes_to_int  # Pass the mapping dict

        # Group images by class (still uses strings internally for balancing)
        class_to_images = defaultdict(list)
        for img_path, label in zip(image_paths, labels):
            class_to_images[label].append(img_path)

        # Balanced sampling logic
        self.final_images = []
        self.final_labels = []
        for cls, paths in class_to_images.items():
            if len(paths) < 100: # oversampling
                sampled_paths = random.choices(paths, k=100)
            elif len(paths) > 200: # subsampling
                sampled_paths = random.sample(paths, k=200)
            else:
                sampled_paths = paths
            self.final_images.extend(sampled_paths)
            self.final_labels.extend([cls] * len(sampled_paths))

        print(f"Balanced dataset constructed with {len(self.final_images)} images across {len(set(self.final_labels))} species.")

    def __len__(self):
        return len(self.final_images)

    def __getitem__(self, idx):
        img_path = self.final_images[idx]
        label = self.final_labels[idx]

        # Convert label to integer
        label = self.classes_to_int[label]  # <--- This converts text to int

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        return {"image": image, "label": torch.tensor(label, dtype=torch.long)}

In [ ]:
# Apply this into your global labels_all array
labels_all = np.copy( unified_species )
labels_all[~filt] = 'background'

In [ ]:
np.unique( labels_all ).shape, np.unique( labels_all, return_counts=True )

In [ ]:
images_all.shape, labels_all.shape

In [ ]:
classes_to_int['background'] = 51

In [ ]:
len(classes_to_int)

## Finetuning

In [ ]:
dataset = BalancedPollenDataset(images_all, labels_all, classes_to_int, transform=transform)

In [ ]:
dataloader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=4)

In [ ]:
# Example ImageNet normalization (adjust if you used different stats)
imagenet_mean = np.array([0.485, 0.456, 0.406])
imagenet_std = np.array([0.229, 0.224, 0.225])

myiter = iter(dataloader)

def denormalize(tensor_img):
    """
    Undo ImageNet normalization and convert tensor to numpy.
    """
    img = tensor_img.cpu().numpy().transpose(1, 2, 0)
    img = img * imagenet_std + imagenet_mean  # Undo normalization
    img = np.clip(img, 0, 1)  # Keep within valid range
    return img

# Fetch one batch (128 images expected)
batch = next(myiter)

# Extract images and labels
images_plot = batch['image']
labels_plot = batch['label']

# Plot 128 images in 16 rows x 8 columns grid
fig, axs = plt.subplots(16, 8, figsize=(16, 32))  # Taller figure to fit all images

for i, ax in enumerate(axs.flat):
    img = denormalize(images_plot[i])
    ax.imshow(img)
    ax.set_title(f"Label: {labels_plot[i].item()}", fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def model_selection(config, num_classes, model_names):
    """
    Select and return a pre-trained model based on the input name or index.

    Parameters:
        config (str | int):
            If str, the name of the model to load.
            If int, the index of the model in the predefined list.
        num_classes (int):
            The number of output classes for the model.

    Returns:
        model (torch.nn.Module): The selected model.
        model_img_input_size (int): The input image size for the model.
    """

    model_num_classes = num_classes+1 # include Unknown class as well !
    print('Model head size:', model_num_classes)

    # Validate and determine the model name
    if isinstance(config, int):
        if config < 0 or config >= len(model_names):
            raise ValueError(f"Index out of range. Valid indices: 0 to {len(model_names) - 1}.")
        model_name = model_names[config]
    elif isinstance(config, str):
        if config not in model_names:
            raise ValueError(f"Model name not recognized. Choose from: {model_names}.")
        model_name = config
    else:
        raise TypeError("Config must be a str or int.")

    # Model selection and configuration
    if "lvd142m" in model_name:
        model = timm.create_model(
            model_name,
            pretrained=True,
            img_size=518,
            init_values=1e-5,
            num_classes=0,  # Remove classifier nn.Linear
        )
        
        if 'small' in model_name:
            in_features_dim = 384
        elif 'base' in model_name:
            in_features_dim = 768
        elif 'large' in model_name:
            in_features_dim = 1024
        
        model.head = nn.Linear(in_features=in_features_dim, out_features=model_num_classes)
        model_img_input_size = 518

    elif model_name == "vit_large_patch16_224":
        model = timm.create_model(
            model_name,
            pretrained=False, # does not need to load !!
            img_size=224,
            init_values=1e-5,
            num_classes=0,  # Remove classifier nn.Linear
        )
        print("Loading pre-trained weights...")
        model.load_state_dict(
            torch.load("../training/vit_large_patch16_224.dinov2_uni_mass100k.pth", map_location="cpu"), strict=True
        )
        model.head = nn.Linear(in_features=1024, out_features=model_num_classes)
        model_img_input_size = 224

    elif model_name == "vit_huge_patch14_224": # "hf-hub:MahmoodLab/UNI2-h":
        # Define the model configuration
        timm_kwargs = {
            'img_size': 224,
            'patch_size': 14,
            'depth': 24,
            'num_heads': 24,
            'init_values': 1e-5,
            'embed_dim': 1536,
            'mlp_ratio': 2.66667 * 2,
            'num_classes': 0,
            'no_embed_class': True,
            'mlp_layer': timm.layers.SwiGLUPacked,
            'act_layer': torch.nn.SiLU,
            'reg_tokens': 8,
            'dynamic_img_size': True
        }
        # Create the model without loading weights from the Hub
        model = timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=False, **timm_kwargs)
        # Load the saved weights
        model.load_state_dict(
            torch.load('./vit_huge_patch14_224.dinov2_uni_mass200k.pth', map_location="cpu"), 
            strict=True
        )
        model.head = nn.Linear(in_features=1536, out_features=model_num_classes)
        model_img_input_size = 224

    elif model_name == "dino_resnet50":
        model = torch.hub.load('facebookresearch/dino:main', 'dino_resnet50', pretrained=True)
        model.fc = nn.Linear(in_features=2048, out_features=model_num_classes)
        model_img_input_size = 224

    elif model_name == "resnet50_coco_pretrained_from_FasterRCNN":
        detection_model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1)
        backbone = detection_model.backbone

        class CustomModel(nn.Module):
            def __init__(self, backbone, num_classes):
                super(CustomModel, self).__init__()
                self.backbone = backbone
                self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))  # Global average pooling
                self.head = nn.Linear(backbone.out_channels, model_num_classes)  # Single linear layer

            def forward(self, x):
                x = self.backbone(x)["0"]  # Forward pass through backbone (assumes single tensor output)
                x = self.global_avg_pool(x)  # Global average pooling
                x = torch.flatten(x, 1)  # Flatten to 1D
                x = self.head(x)  # Linear classification layer
                return x

        model = CustomModel(backbone=backbone, num_classes=model_num_classes)
        model_img_input_size = 224

    else:
        raise ValueError(f"Unsupported model: {model_name}")

    # Freeze parameters except the classifier head
    for param in model.parameters():
        param.requires_grad = False

    if hasattr(model, 'head'):
        for param in model.head.parameters():
            param.requires_grad = True
    elif hasattr(model, 'fc'):
        for param in model.fc.parameters():
            param.requires_grad = True

    print('Model input image size:', model_img_input_size)
    return model, model_img_input_size


def seed_torch(CUDANUM, seed=7):
    import random
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    device=torch.device(f'cuda:{CUDANUM}' if torch.cuda.is_available() else "cpu") 
    if device.type == f'cuda:{CUDANUM}':
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # if you are using multi-GPU.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

In [ ]:
cuda_num = "0"
seed_torch(42)

# List of all possible model names
model_names = [
    "vit_large_patch16_224",
    "vit_small_patch14_dinov2.lvd142m",
    "vit_base_patch14_dinov2.lvd142m",
    "vit_large_patch14_dinov2.lvd142m",
    "dino_resnet50",
    "resnet50_coco_pretrained_from_FasterRCNN",
    "vit_huge_patch14_224"
]
model_num = 1  # Index for selecting the model from the list

In [ ]:
## MODEL SELECTION
device = torch.device(f'cuda:{cuda_num}') if torch.cuda.is_available() else torch.device('cpu')

print('Selected model:', model_names[model_num])
model, model_img_input_size = model_selection(model_num, len(classes_to_int)-1, model_names)
model.to(device);

In [ ]:
# Splitting function
def split_train_val(image_paths, labels, val_ratio=0.2, seed=42):
    random.seed(seed)
    species_to_images = defaultdict(list)
    for img_path, label in zip(image_paths, labels):
        species_to_images[label].append(img_path)

    train_paths, val_paths, train_labels, val_labels = [], [], [], []
    for label, paths in species_to_images.items():
        random.shuffle(paths)
        split_idx = int(len(paths) * (1 - val_ratio))
        train_paths.extend(paths[:split_idx])
        val_paths.extend(paths[split_idx:])
        train_labels.extend([label] * split_idx)
        val_labels.extend([label] * (len(paths) - split_idx))

    return (train_paths, train_labels), (val_paths, val_labels)

In [ ]:
len(labels_all)

In [ ]:
# Split into train and validation
(train_paths, train_labels), (val_paths, val_labels) = split_train_val(images_all, labels_all)

In [ ]:
class CLAHETransform:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    def __call__(self, img):
        img_np = np.array(img)
        if len(img_np.shape) == 3:
            img_gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        else:
            img_gray = img_np
        img_clahe = self.clahe.apply(img_gray)
        img_clahe = cv2.cvtColor(img_clahe, cv2.COLOR_GRAY2RGB)  # Convert back to 3-channel for the rest of the pipeline
        return Image.fromarray(img_clahe)

train_transform = transforms.Compose([
    transforms.Resize((518, 518)),
    CLAHETransform(),
    transforms.RandomGrayscale(p=0.2),  # 20% chance to remove color
    transforms.RandomResizedCrop(size=518, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.GaussianBlur(kernel_size=(3, 5), sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

In [ ]:
val_transform = transforms.Compose(
    [
    transforms.Resize((518, 518)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)) # IMAGENET !
    ])

In [ ]:
# Fixed validation set (doesn't change across epochs)
val_dataset = PollenValidationDataset(val_paths, val_labels, classes_to_int, transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=True, num_workers=4)

In [ ]:
seed_torch(CUDANUM=cuda_num, seed=42) # FIX SEED 

In [ ]:
backbone_lr = 5e-6   # Slow learning rate for pretrained backbone
head_lr = 1e-4       # Faster learning rate for classifier head

# Example: Freeze all backbone except head
for name, param in model.named_parameters():
    if 'head' in name:  # Adapt to actual head layer name if needed
        param.requires_grad = True
    else:
        param.requires_grad = False  # Freezing backbone

In [ ]:
num_epochs = 100
best_val_acc = 0.0
best_val_loss = float("inf")  # Initialize before training loop
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

In [ ]:
param_groups = [
    {"params": [p for n, p in model.named_parameters() if 'head' in n], "lr": head_lr},
    {"params": [p for n, p in model.named_parameters() if 'head' not in n], "lr": backbone_lr}
]
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(param_groups, weight_decay=1e-4)

lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

In [ ]:
scaler = GradScaler()

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    
    # Re-sample train set with new balance for each epoch
    train_dataset = BalancedPollenDataset(
        train_paths, train_labels, classes_to_int,
        transform=train_transform, seed=epoch*42+1337
    )
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=16)

    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch in tqdm(train_loader):
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad(set_to_none=True)

        # ------- AMP forward & backward -------
        with autocast():                          # activations in fp16/bf16
            outputs = model(images)
            loss    = criterion(outputs, labels)
        
        scaler.scale(loss).backward()             # backward in fp16
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)                    # optimizer step in fp32
        scaler.update()                           # adjust scaling factor
        
        # --------------------------------------
        
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    train_acc = correct / total

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)

    print(f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.2%}")

    if epoch == 24:  # Unfreeze backbone at epoch 24
        print("Unfreezing backbone for fine-tuning!")
        for name, param in model.named_parameters():
            param.requires_grad = True
        # You can also reset the optimizer with new LR setup here if needed


    # ---- VALIDATION ----
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(val_loader):
            images = batch["image"].to(device)
            labels = batch["label"].to(device)

            with autocast():                      # fp16 inference
                outputs = model(images)
                loss    = criterion(outputs, labels)
                
            #outputs = model(images)
            #loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    val_loss = running_loss / len(val_loader.dataset)
    val_acc = correct / total

    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.2%}")

    # Adaptive LR Update
    lr_scheduler.step()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
    
        # Save model
        model_path = "newrun_vit_small_lvd_best_model_val_loss.pth"
        torch.save(model.state_dict(), model_path)
        print(f"Saved best model (val_loss={val_loss:.4f}) to {model_path}")

print(f"Training Complete! Best Val Loss: {best_val_loss:.2%}")

In [ ]:
import pandas as pd

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# Save training history to CSV
history_df = pd.DataFrame(history)
csv_path = f"history_vit_small_lvd_{timestamp}.csv"
history_df.to_csv(csv_path, index_label="epoch")
print(f"Training history saved to {csv_path}")

In [ ]:
best_val_loss